In [1]:
import tensorflow as tf

# 시스템에 설치된 GPU 장치 리스트 출력
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"GPU 사용 가능: {gpus}")
else:
    print("GPU가 사용되지 않거나 설치되지 않았습니다.")

GPU가 사용되지 않거나 설치되지 않았습니다.


In [ ]:
import torch

# GPU 사용 가능 여부 확인
if torch.cuda.is_available():
    print(f"GPU 사용 가능: {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 사용되지 않거나 설치되지 않았습니다.")

## vLLM 서버 설치 및 실행
아래 셀들을 순서대로 실행하세요.

In [ ]:
# vLLM 및 웹 UI용 패키지 설치 (test 커널 환경)
import sys
!{sys.executable} -m pip install vllm gradio openai --quiet

In [ ]:
import subprocess
import time
import sys
import os

# ─────────────────────────────────────────────
# 설정: 사용할 모델 경로 또는 HuggingFace 모델 ID
# 로컬 GGUF 파일이 아닌 HF 모델을 쓰려면 모델 ID로 변경하세요.
# 예) MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
MODEL = "facebook/opt-125m"   # 테스트용 경량 모델 (GPU 없어도 동작)
HOST  = "0.0.0.0"             # 외부 접근 허용 (로컬만 원하면 127.0.0.1)
PORT  = 8000
# ─────────────────────────────────────────────

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--host", HOST,
    "--port", str(PORT),
    "--dtype", "float16",          # GPU 없으면 float32 로 변경
    "--max-model-len", "2048",
]

print(f"vLLM 서버 시작 중...\n명령어: {' '.join(cmd)}")
vllm_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# 서버가 준비될 때까지 로그 출력
for line in vllm_proc.stdout:
    print(line, end="")
    if "Application startup complete" in line or "Uvicorn running" in line:
        print(f"\n✅ vLLM 서버 준비 완료! → http://localhost:{PORT}/docs")
        break

In [ ]:
import gradio as gr
from openai import OpenAI

# vLLM OpenAI 호환 엔드포인트에 연결
client = OpenAI(
    base_url=f"http://localhost:{PORT}/v1",
    api_key="dummy",  # vLLM은 키 불필요
)

def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=512,
        stream=True,
    )

    partial = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        partial += delta
        yield partial

demo = gr.ChatInterface(
    fn=chat,
    title="vLLM Chat",
    description=f"Model: {MODEL}  |  Server: http://localhost:{PORT}",
)

# share=True 로 설정하면 외부 공개 URL(gradio.live)도 발급됩니다
demo.launch(server_name="0.0.0.0", server_port=7860, share=True)

In [ ]:
# 서버 종료가 필요할 때 이 셀을 실행하세요
try:
    vllm_proc.terminate()
    print("vLLM 서버가 종료되었습니다.")
except NameError:
    print("실행 중인 vLLM 프로세스가 없습니다.")